In [1]:
import pandas as pd
import numpy as np
import glob
import os
import dask.dataframe as dd
import re

In [2]:
from dask.distributed import Client
client = Client(n_workers=8, threads_per_worker=2, memory_limit='200GB')

/home/kpegion/miniconda3/envs/s2stransportation/lib/python3.13/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 36467 instead
  warnings.warn(
Task exception was never retrieved
future: <Task finished name='Task-1436' coro=<Client._gather.<locals>.wait() done, defined at /home/kpegion/miniconda3/envs/s2stransportation/lib/python3.13/site-packages/distributed/client.py:2385> exception=AllExit()>
Traceback (most recent call last):
  File "/home/kpegion/miniconda3/envs/s2stransportation/lib/python3.13/site-packages/distributed/client.py", line 2394, in wait
    raise AllExit()
distributed.client.AllExit
2025-10-28 14:24:33,611 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:34475'.
2025-10-28 14:24:33,613 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:41461'.
2025-10-28 14:24:33,615 - 

input_files='../data/swe/csv/daily/us_ssmv11034tS__T0001TTNATS*.csv'
all_files = glob.glob(input_files)

ddf = dd.read_csv("../data/swe/csv/daily/us_ssmv11034tS__T0001TTNATS*.csv",header=0)
ddf.head()

# Group by Year and Month, compute average, treating -9999 as missing on the fly

# Filter out invalid values first
ddf_valid = ddf[ddf['Value'] != -9999]

# Now groupby Year and Month and compute mean
monthly_avg_df = (
    ddf_valid.groupby(['Year', 'Month'])['Value']
             .mean()
             .reset_index()
             .rename(columns={'Value':'Avg_Value'})
#             .compute()
)

print(monthly_avg_df.head())


# Directory with daily CSVs
input_path = "../data/swe/csv/daily/"
#output_path = "../data/swe/csv/monthly_avg/"

#os.makedirs(output_path, exist_ok=True)

# 1️⃣ Read all CSVs lazily
ddf = dd.read_csv(os.path.join(input_path, "us_ssmv11034tS__T0001TTNATS*.csv"), header=0)

# 2️⃣ Replace invalid SWE values (-9999) with NaN
ddf['Value'] = ddf['Value'].mask(ddf['Value'] == -9999, np.nan)

# 3️⃣ Optional: convert Year and Month to categorical to speed up groupby
ddf['Year'] = ddf['Year'].astype('category')
ddf['Month'] = ddf['Month'].astype('category')

print("Computing monthly averages")
# 4️⃣ Compute monthly averages per grid cell (X,Y)
monthly_avg = (
    ddf.groupby(['Year','Month','X','Y'])['Value']
       .mean(split_out=25)  # parallelize aggregation
       .reset_index()
       .rename(columns={'Value':'Avg_Value'})
)

# 5️⃣ Persist in memory (optional, speeds up repeated operations)
monthly_avg = monthly_avg.persist()

# 6️⃣ Compute the result as pandas DataFrame
monthly_avg_df = monthly_avg.compute()

# 7️⃣ Save to CSV per month/year if desired
for (year, month), group in monthly_avg_df.groupby(['Year','Month']):
    out_file = os.path.join(output_path, f"snow_water_equiv_{year}_{month:02d}_avg.csv")
    group.to_csv(out_file, index=False)
    print(f"Saved {out_file}")

# Read all daily SWE CSVs in parallel
ddf = dd.read_csv("../data/swe/csv/daily/us_ssmv11034tS__T0001TTNATS*.csv", header=0)

# Replace missing flag values with NaN
ddf["Value"] = ddf["Value"].replace(-9999, np.nan)

# Compute monthly mean per grid cell
monthly_avg = (
    ddf.groupby(["Year", "Month", "X", "Y"])
       .agg({"Value": "mean"})
       .reset_index()
)

# Compute in parallel and bring to memory (may take time but less than daily)
monthly_avg_df = monthly_avg.compute()

# Build a datetime index for monthly means
monthly_avg_df["time"] = dd.to_datetime(
    monthly_avg_df["Year"].astype(str) + "-" + 
    monthly_avg_df["Month"].astype(str).str.zfill(2) + "-15"
)

# Convert to xarray Dataset
ds = monthly_avg_df.pivot_table(index="time", columns=["Y", "X"], values="Value").to_xarray()


# Read all daily SWE CSVs and include the file path for date extraction
ddf = dd.read_csv(
    "../data/swe/csv/daily/us_ssmv11034tS__T0001TTNATS*.csv",
    header=0,
    include_path_column=True
)

# Replace missing values
ddf["Value"] = ddf["Value"].replace(-9999, np.nan)

# Extract date (YYYYMMDD) from each file name and convert to datetime
ddf["time"] = ddf["path"].map_partitions(
    lambda s: s.apply(lambda x: pd.to_datetime(re.search(r"(\d{8})", x).group(1), format="%Y%m%d"))
)

# (optional) check sample
print(ddf[["path", "time"]].head())

# Bring to memory (optional — or continue lazily)
df = ddf.compute()

print(df[["X", "Y", "Value", "time"]].head())


import dask.dataframe as dd
import numpy as np
import dask

# Read all daily SWE CSVs
ddf = dd.read_csv(
    "../data/swe/csv/daily/us_ssmv11034tS__T0001TTNATS*.csv",
    header=0,
    dtype={"Year": int, "Month": int, "Day": int, "X": float, "Y": float, "Value": float}
)

ddf["Value"] = ddf["Value"].replace(-9999, np.nan)

# Create datetime (still lazy)
ddf["time"] = dd.to_datetime(
    ddf["Year"].astype(str) + "-" +
    ddf["Month"].astype(str).str.zfill(2) + "-" +
    ddf["Day"].astype(str).str.zfill(2)
)

# Compute monthly mean (optional)
# monthly_avg = ddf.groupby(["Year", "Month", "X", "Y"])["Value"].mean().reset_index()

# Persist to avoid recomputation
ddf = ddf.persist()

# Pivot in smaller chunks
def pivot_chunk(df):
    return df.pivot_table(index="time", columns=["Y", "X"], values="Value")

# Apply lazily across partitions
meta = pd.DataFrame(columns=pd.MultiIndex.from_tuples([(0,0)], names=["Y","X"]), index=pd.to_datetime([]))
ddf_grouped = ddf.map_partitions(pivot_chunk, meta=meta)

# Compute final result (this is the only big step)
df_pivot = ddf_grouped.compute()

# Convert to xarray
ds = df_pivot.to_xarray()


In [3]:
import dask.dataframe as dd
import xarray as xr
import pandas as pd
import re
from glob import glob

# 1️⃣ List files
files = sorted(glob("../data/swe/csv/daily/us_ssmv11034tS__T0001TTNATS*.csv"))

# 2️⃣ Read all CSVs in parallel
ddf = dd.read_csv(files, header=0, include_path_column="path")

# 3️⃣ Replace missing flags
ddf["Value"] = ddf["Value"].replace(-9999, float("nan"))

# 4️⃣ Extract date from filename
ddf["time"] = ddf["path"].map_partitions(
    lambda s: s.apply(lambda x: pd.to_datetime(re.search(r"(\d{8})", x).group(1), format="%Y%m%d")),
    meta=("time", "datetime64[ns]")
)

ddf = ddf.drop(columns="path")

# 5️⃣ Compute the Dask DataFrame in chunks
df = ddf.compute()  # if memory allows; else use smaller partitions

# 6️⃣ Convert to xarray using pivot_table
# This converts (time, Y, X) -> 3D array
ds = df.pivot_table(index="time", columns=["Y","X"], values="Value").to_xarray()

# 7️⃣ Rename the variable to SWE
ds = ds.rename({"Value": "SWE"})

# 8️⃣ Swap dimensions to standard order (time, Y, X)
ds = ds.transpose("time", "Y", "X")

print(ds)


2025-10-28 14:24:28,585 - distributed.worker - ERROR - 
Traceback (most recent call last):
  File "/home/kpegion/miniconda3/envs/s2stransportation/lib/python3.13/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ~~~~~~~~~~^^^^^^
  File "/home/kpegion/miniconda3/envs/s2stransportation/lib/python3.13/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/home/kpegion/miniconda3/envs/s2stransportation/lib/python3.13/asyncio/base_events.py", line 712, in run_until_complete
    self.run_forever()
    ~~~~~~~~~~~~~~~~^^
  File "/home/kpegion/miniconda3/envs/s2stransportation/lib/python3.13/asyncio/base_events.py", line 683, in run_forever
    self._run_once()
    ~~~~~~~~~~~~~~^^
  File "/home/kpegion/miniconda3/envs/s2stransportation/lib/python3.13/asyncio/base_events.py", line 2004, in _run_once
    event_list = self._selector.select(timeout)
  File "/home/kpegion/miniconda3/env